# 🔗 Day 4 — Pipelines, Gradio & Deployment
### AI Workshop · Google Colab Notebook

---

**Today you will:**
- Build a Gradio web UI around any Python function in < 10 lines
- Chain two models together into a multi-model pipeline
- Wrap the pipeline in a real web interface
- Deploy to HuggingFace Spaces — a permanent public URL

**The pipeline you build today:**
```
Image Upload
     ↓
[YOLO Detector]  →  object labels (person, car, dog...)
     ↓
[LLM]  →  scene description / story / safety advice
     ↓
Gradio UI  →  annotated image + text output
     ↓
HuggingFace Spaces  →  public URL, shareable with anyone
```

> ⚠️ **Runtime:** Go to **Runtime → Change runtime type → T4 GPU → Save** before running Section 0.

> 📋 Run cells **in order**. If runtime resets, re-run from Section 0.

---
## Section 0 — Setup
Install everything first. Takes ~90 seconds.

In [ ]:
!pip install gradio ultralytics transformers accelerate huggingface_hub -q

import gradio as gr
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO
from transformers import pipeline
import requests, os

device = 0 if torch.cuda.is_available() else -1
print(f'✅ Setup complete — running on: {"GPU" if device == 0 else "CPU"}')
print(f'   Gradio version: {gr.__version__}')

In [ ]:
# Load models — do this once at the top, not inside your functions
# Loading inside a function means reloading every time → very slow

# ── Model 1: YOLOv8 object detector ──────────────────────────────────────────
# Option A: Use your trained model from Day 3
# Option B: Use the pre-trained COCO model (works with any image)

print('Loading YOLO model...')

# Change to your Day 3 model path if you have it:
# yolo_model = YOLO('workshop_runs/my_detector/weights/best.pt')
yolo_model = YOLO('yolov8n.pt')   # pre-trained COCO (fallback)

print(f'✅ YOLO ready — knows {len(yolo_model.names)} classes')

# ── Model 2: TinyLlama LLM ────────────────────────────────────────────────────
print('\nLoading TinyLlama... (~2 min first time)')
llm_pipe = pipeline(
    'text-generation',
    model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    torch_dtype=torch.float16 if device == 0 else torch.float32,
    device=device,
)
print('✅ LLM ready')
print('\n🚀 All models loaded. Move to Section 1.')

---
## Section 1 — 🔵 Gradio: Your First App

Gradio wraps **any Python function** into a web interface.
The function takes inputs → does something → returns outputs.
Gradio handles everything else: the UI, the server, the URL.

In [ ]:
# ── The simplest possible Gradio app — 5 lines ────────────────────────────────
import gradio as gr

def greet(name):
    return f'Hello, {name}! Welcome to Day 4 of the AI Workshop.'

demo = gr.Interface(
    fn      = greet,
    inputs  = gr.Textbox(label='Your name'),
    outputs = gr.Textbox(label='Greeting'),
    title   = 'My First Gradio App',
)

demo.launch(share=True)   # share=True → public gradio.live URL
# 🌐 Copy the public URL and open it on your phone!

In [ ]:
demo.close()   # close before launching the next one

In [ ]:
# ── Gradio components — the building blocks ───────────────────────────────────
# These are the input/output types you'll use most

print('GRADIO INPUT COMPONENTS:')
components = {
    'gr.Textbox()':    'Text input / output',
    'gr.Image()':      'Image upload / display',
    'gr.Slider()':     'Number slider',
    'gr.Dropdown()':   'Select from list',
    'gr.Checkbox()':   'True/False toggle',
    'gr.File()':       'File upload',
    'gr.Audio()':      'Audio record / play',
    'gr.Chatbot()':    'Chat message history',
    'gr.Number()':     'Numeric input',
    'gr.Radio()':      'Radio button selection',
}
for comp, desc in components.items():
    print(f'  {comp:<22} → {desc}')

print('\n💡 A function can have multiple inputs and multiple outputs.')
print('   Just pass lists to inputs= and outputs=')

In [ ]:
# ── Gradio with image input/output ────────────────────────────────────────────
# This is the pattern you'll use for your AI pipeline

def flip_image(image):
    """Flip an image horizontally — simple demo with image I/O."""
    if image is None:
        return None
    img = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    return img.transpose(Image.FLIP_LEFT_RIGHT)

demo2 = gr.Interface(
    fn          = flip_image,
    inputs      = gr.Image(label='Upload an image'),
    outputs     = gr.Image(label='Flipped image'),
    title       = 'Image Flipper',
    description = 'Upload any image and see it flipped horizontally.',
)
demo2.launch(share=True)

In [ ]:
demo2.close()

---
## Section 2 — 🟡 Exercise: Single-Model App

Wrap your YOLO model in a Gradio interface.

Input: an image  
Output: the same image with bounding boxes drawn + a text list of what was detected

In [ ]:
# ── YOLO Gradio App ───────────────────────────────────────────────────────────

def detect_objects(image, confidence_threshold=0.25):
    """
    Run YOLO detection on an uploaded image.
    Returns: annotated image, detection summary text.
    """
    if image is None:
        return None, 'Please upload an image.'

    # Ensure PIL Image format
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    # Run detection
    results = yolo_model.predict(
        source  = image,
        conf    = confidence_threshold,
        verbose = False,
    )

    # Draw boxes on image
    annotated = results[0].plot()          # returns BGR numpy array
    annotated_rgb = annotated[:, :, ::-1]  # BGR → RGB
    output_img = Image.fromarray(annotated_rgb)

    # Build text summary
    boxes = results[0].boxes
    names = results[0].names

    if len(boxes) == 0:
        summary = f'No objects detected above {confidence_threshold:.0%} confidence.'
    else:
        lines = [f'Detected {len(boxes)} object(s):\n']
        for box in boxes:
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            lines.append(f'  • {names[cls]:<20} {conf:.1%}')
        summary = '\n'.join(lines)

    return output_img, summary


# Build the Gradio interface
yolo_app = gr.Interface(
    fn      = detect_objects,
    inputs  = [
        gr.Image(label='Upload image'),
        gr.Slider(minimum=0.1, maximum=0.9, value=0.25, step=0.05,
                  label='Confidence threshold'),
    ],
    outputs = [
        gr.Image(label='Detected objects'),
        gr.Textbox(label='Detection summary', lines=8),
    ],
    title       = '🎯 Object Detector',
    description = 'Upload any image. YOLOv8 will detect and label objects.',
    examples    = [['https://ultralytics.com/images/bus.jpg', 0.25]]
        if False else None,  # set True if you have example images
)

yolo_app.launch(share=True)
print('\n👆 Open the public URL on your phone and share it with the person next to you!')

In [ ]:
yolo_app.close()

---
## Section 3 — 🔗 Multi-Model Pipeline

Now we chain YOLO + LLM together.

```
Image → [YOLO] → detected labels → [LLM] → response
```

The LLM gets the object labels as context and generates a response based on a prompt you write.

In [ ]:
# ── Helper: call the LLM ──────────────────────────────────────────────────────

def call_llm(prompt, system_prompt='You are a helpful assistant.', temperature=0.7):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': prompt},
    ]
    formatted = llm_pipe.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    result = llm_pipe(
        formatted,
        max_new_tokens=200,
        temperature=max(float(temperature), 0.01),
        do_sample=True,
        pad_token_id=llm_pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )
    return result[0]['generated_text'].strip()

print('✅ call_llm() ready')

In [ ]:
# ── The complete pipeline function ────────────────────────────────────────────

PIPELINE_MODES = {
    'Scene Description':  'You are a descriptive narrator. Describe the scene based on the detected objects in 2-3 sentences.',
    'Safety Advisor':     'You are a safety expert. Based on the objects detected, identify any potential safety concerns in 2-3 bullet points.',
    'Fun Facts':          'You are an enthusiastic educator. Share one fun or surprising fact about the most interesting object detected.',
    'Instagram Caption':  'You are a creative social media writer. Write a catchy Instagram caption based on what is in the scene. Include 3 relevant hashtags.',
    'Story Starter':      'You are a creative writer. Begin a short story (2-3 sentences) inspired by the objects detected in the scene.',
}

def vision_language_pipeline(image, confidence=0.25, mode='Scene Description'):
    """
    Full pipeline: Image → YOLO → LLM → response.

    Parameters
    ----------
    image      : uploaded image (PIL or numpy)
    confidence : YOLO confidence threshold
    mode       : which LLM prompt to use

    Returns
    -------
    annotated_image : image with bounding boxes
    llm_response    : text from the LLM
    detection_info  : raw detection summary
    """
    if image is None:
        return None, 'Please upload an image.', ''

    # ── Step 1: YOLO detection ────────────────────────────────────────────────
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    results  = yolo_model.predict(source=image, conf=confidence, verbose=False)
    result   = results[0]
    boxes    = result.boxes
    names    = result.names

    # Draw boxes
    annotated     = result.plot()[:, :, ::-1]  # BGR→RGB
    annotated_img = Image.fromarray(annotated)

    # Extract labels and confidence scores
    if len(boxes) == 0:
        detection_text = 'No objects detected.'
        labels_str     = 'nothing'
    else:
        detections     = [(names[int(b.cls[0])], float(b.conf[0])) for b in boxes]
        detection_text = '\n'.join([f'• {n}: {c:.0%}' for n, c in detections])
        labels_str     = ', '.join([n for n, _ in detections])

    # ── Step 2: LLM generation ────────────────────────────────────────────────
    system_prompt = PIPELINE_MODES.get(mode, PIPELINE_MODES['Scene Description'])
    user_prompt   = f'Objects detected in the image: {labels_str}.'

    llm_response = call_llm(
        prompt        = user_prompt,
        system_prompt = system_prompt,
        temperature   = 0.7,
    )

    return annotated_img, llm_response, detection_text


print('✅ Pipeline function ready.')
print('   Inputs:  image, confidence slider, mode dropdown')
print('   Outputs: annotated image, LLM response, detection list')

In [ ]:
# ── Test the pipeline without Gradio first ────────────────────────────────────
# Always test your function before wrapping it in a UI

test_url = 'https://ultralytics.com/images/bus.jpg'
response  = requests.get(test_url)
test_img  = Image.open(__import__('io').BytesIO(response.content))

print('Running pipeline test...')
ann_img, llm_text, det_text = vision_language_pipeline(
    image      = test_img,
    confidence = 0.25,
    mode       = 'Scene Description'
)

print('\n📊 YOLO detected:')
print(det_text)
print('\n🤖 LLM response:')
print(llm_text)

# Display the annotated image
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
plt.imshow(ann_img)
plt.title('Pipeline output — annotated image')
plt.axis('off')
plt.show()

print('\n✅ Pipeline test passed! Build the Gradio UI next.')

In [ ]:
# ── Wrap the pipeline in Gradio ───────────────────────────────────────────────

with gr.Blocks(title='🔗 Vision + Language Pipeline') as pipeline_app:

    gr.Markdown('# 🔗 Vision + Language Pipeline')
    gr.Markdown('Upload an image → YOLO detects objects → LLM responds based on what it sees.')

    with gr.Row():
        # ── Left column: inputs ───────────────────────────────────────────────
        with gr.Column(scale=1):
            image_input = gr.Image(label='📷 Upload image')
            conf_slider = gr.Slider(
                minimum=0.1, maximum=0.9, value=0.25, step=0.05,
                label='Detection confidence'
            )
            mode_dropdown = gr.Dropdown(
                choices=list(PIPELINE_MODES.keys()),
                value='Scene Description',
                label='🎭 LLM Mode',
            )
            run_btn = gr.Button('🚀 Run Pipeline', variant='primary')

        # ── Right column: outputs ─────────────────────────────────────────────
        with gr.Column(scale=1):
            image_output = gr.Image(label='🎯 Detected objects')
            llm_output   = gr.Textbox(label='🤖 LLM Response', lines=6)
            det_output   = gr.Textbox(label='📊 Detections', lines=5)

    run_btn.click(
        fn      = vision_language_pipeline,
        inputs  = [image_input, conf_slider, mode_dropdown],
        outputs = [image_output, llm_output, det_output],
    )

    gr.Markdown('---')
    gr.Markdown('**Try different modes** — same image, completely different outputs based on the LLM prompt.')

pipeline_app.launch(share=True)
print('\n🌐 Your pipeline app is live! Share the URL.')

In [ ]:
pipeline_app.close()

---
## Section 4 — 🟡 Customise Your Pipeline

Now make it your own. Change the prompt, change the output, change the logic.

**Ideas to try:**
- Add a new LLM mode to `PIPELINE_MODES`
- Add a confidence filter: only pass objects above 60% to the LLM
- Count objects by class and show a count summary
- Add a language selector: respond in Nepali, Spanish, French, etc.
- Add a chat history so users can ask follow-up questions

In [ ]:
# 🟡 YOUR CUSTOM PIPELINE
# Modify the pipeline function or add new modes below
# ─────────────────────────────────────────────────────────────────────────────

# Add your custom LLM modes here:
MY_MODES = {
    'Custom Mode 1': 'Your system prompt for mode 1 here.',
    'Custom Mode 2': 'Your system prompt for mode 2 here.',
    # Add more...
}

def my_custom_pipeline(image, confidence=0.25, mode='Custom Mode 1', language='English'):
    """
    Your custom pipeline. Modify this function to change what it does.
    """
    if image is None:
        return None, 'Please upload an image.', ''

    # ── Step 1: Detection ─────────────────────────────────────────────────────
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    results       = yolo_model.predict(source=image, conf=confidence, verbose=False)
    result        = results[0]
    annotated     = Image.fromarray(result.plot()[:, :, ::-1])

    boxes = result.boxes
    names = result.names

    if len(boxes) == 0:
        return annotated, 'No objects detected.', 'Nothing found.'

    detections = [(names[int(b.cls[0])], float(b.conf[0])) for b in boxes]
    labels_str = ', '.join([n for n, c in detections if c >= 0.5])  # filter by confidence

    if not labels_str:
        labels_str = 'some unidentified objects'

    # ── Step 2: LLM ──────────────────────────────────────────────────────────
    sys_prompt = MY_MODES.get(mode, 'You are a helpful assistant.')
    # Add language instruction:
    if language != 'English':
        sys_prompt += f' Always respond in {language}.'

    user_prompt = f'Objects detected: {labels_str}.'
    response    = call_llm(user_prompt, sys_prompt, temperature=0.7)

    det_text = '\n'.join([f'• {n}: {c:.0%}' for n, c in detections])
    return annotated, response, det_text


# ─────────────────────────────────────────────────────────────────────────────

with gr.Blocks(title='My Custom AI Pipeline') as my_app:
    gr.Markdown('# 🛠️ My Custom AI Pipeline')
    with gr.Row():
        with gr.Column():
            img_in   = gr.Image(label='Upload image')
            conf_in  = gr.Slider(0.1, 0.9, 0.25, step=0.05, label='Confidence')
            mode_in  = gr.Dropdown(choices=list(MY_MODES.keys()), label='Mode')
            lang_in  = gr.Dropdown(
                choices=['English','Nepali','Spanish','French','Hindi','Japanese'],
                value='English', label='Response language'
            )
            btn      = gr.Button('Run', variant='primary')
        with gr.Column():
            img_out  = gr.Image(label='Detections')
            llm_out  = gr.Textbox(label='Response', lines=6)
            det_out  = gr.Textbox(label='Detected objects', lines=5)

    btn.click(fn=my_custom_pipeline,
              inputs=[img_in, conf_in, mode_in, lang_in],
              outputs=[img_out, llm_out, det_out])

my_app.launch(share=True)

In [ ]:
my_app.close()

---
## Section 5 — 🚀 Deploy to HuggingFace Spaces

Your app runs in Colab — but Colab shuts down. HuggingFace Spaces gives you a **permanent public URL**.

### What you need:
1. A HuggingFace account (huggingface.co)
2. A new Space: huggingface.co/new-space → SDK: Gradio → Public
3. Three files: `app.py`, `requirements.txt`, (optional) `README.md`

### Step 1 — Generate your app.py

In [ ]:
# This cell writes a self-contained app.py for HF Spaces
# It downloads the models at startup (no local file needed)

app_py_content = '''
# app.py — HuggingFace Spaces deployment
# Vision + Language Pipeline

import gradio as gr
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO
from transformers import pipeline as hf_pipeline

# ── Load models at startup ────────────────────────────────────────────────────
device     = 0 if torch.cuda.is_available() else -1
dtype      = torch.float16 if device == 0 else torch.float32

print("Loading YOLO...")
yolo_model = YOLO("yolov8n.pt")   # downloads automatically

print("Loading LLM...")
llm_pipe = hf_pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=dtype,
    device=device,
)
print("✅ Models ready")

MODES = {
    "Scene Description": "You are a descriptive narrator. Describe the scene in 2-3 sentences.",
    "Safety Advisor":    "You are a safety expert. Identify potential hazards in bullet points.",
    "Fun Facts":         "Share one fun fact about the most interesting detected object.",
    "Instagram Caption": "Write a catchy Instagram caption with 3 hashtags.",
}

def call_llm(prompt, system_prompt, temperature=0.7):
    messages  = [{"role": "system", "content": system_prompt},
                 {"role": "user",   "content": prompt}]
    formatted = llm_pipe.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipe(formatted, max_new_tokens=200, temperature=max(temperature, 0.01),
                   do_sample=True, pad_token_id=llm_pipe.tokenizer.eos_token_id,
                   return_full_text=False)
    return out[0]["generated_text"].strip()

def run_pipeline(image, confidence=0.25, mode="Scene Description"):
    if image is None:
        return None, "Please upload an image.", ""
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    results   = yolo_model.predict(source=image, conf=confidence, verbose=False)
    result    = results[0]
    annotated = Image.fromarray(result.plot()[:, :, ::-1])

    boxes = result.boxes
    if len(boxes) == 0:
        return annotated, "No objects detected.", "Nothing found."

    names      = result.names
    detections = [(names[int(b.cls[0])], float(b.conf[0])) for b in boxes]
    labels_str = ", ".join([n for n, _ in detections])
    det_text   = "\\n".join([f"• {n}: {c:.0%}" for n, c in detections])

    sys_prompt = MODES.get(mode, MODES["Scene Description"])
    response   = call_llm(f"Objects detected: {labels_str}.", sys_prompt)

    return annotated, response, det_text

with gr.Blocks(title="Vision + Language Pipeline") as demo:
    gr.Markdown("# 🔗 Vision + Language Pipeline")
    gr.Markdown("Upload an image → YOLO detects objects → LLM responds.")
    with gr.Row():
        with gr.Column():
            img_in  = gr.Image(label="Upload image")
            conf_in = gr.Slider(0.1, 0.9, 0.25, step=0.05, label="Confidence")
            mode_in = gr.Dropdown(choices=list(MODES.keys()), value="Scene Description", label="Mode")
            btn     = gr.Button("Run Pipeline", variant="primary")
        with gr.Column():
            img_out = gr.Image(label="Detections")
            llm_out = gr.Textbox(label="LLM Response", lines=6)
            det_out = gr.Textbox(label="Detected objects", lines=5)
    btn.click(fn=run_pipeline, inputs=[img_in, conf_in, mode_in],
              outputs=[img_out, llm_out, det_out])

demo.launch()
'''

with open('app.py', 'w') as f:
    f.write(app_py_content.strip())

print('✅ app.py written')

In [ ]:
# Step 2 — Write requirements.txt

requirements = """gradio>=4.0.0
ultralytics>=8.0.0
transformers>=4.35.0
accelerate>=0.24.0
torch>=2.0.0
Pillow>=9.0.0
numpy>=1.24.0
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('✅ requirements.txt written')
print(requirements)

In [ ]:
# Step 3 — Upload to HuggingFace Spaces
# Replace YOUR_USERNAME and YOUR_SPACE_NAME below
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import HfApi

HF_USERNAME  = 'versey'   # ← your HuggingFace username
SPACE_NAME   = 'vision-language-pipeline'  # ← your Space name (create it first)
HF_TOKEN     = ''  # ← your HF write token

# ─────────────────────────────────────────────────────────────────────────────

if 'YOUR_HF_USERNAME' not in HF_USERNAME and 'YOUR_TOKEN' not in HF_TOKEN:
    api      = HfApi(token=HF_TOKEN)
    repo_id  = f'{HF_USERNAME}/{SPACE_NAME}'

    for filename in ['app.py', 'requirements.txt']:
        api.upload_file(
            path_or_fileobj = filename,
            path_in_repo    = filename,
            repo_id         = repo_id,
            repo_type       = 'space',
        )
        print(f'✅ Uploaded {filename}')

    print(f'\n🚀 Deploying... check your Space at:')
    print(f'   https://huggingface.co/spaces/{repo_id}')
    print(f'\n   Public app URL (after build ~2 min):')
    print(f'   https://{HF_USERNAME}-{SPACE_NAME}.hf.space')
else:
    print('⚠️  Replace YOUR_HF_USERNAME, SPACE_NAME, and HF_TOKEN above.')
    print('\nAlternative: drag-and-drop app.py and requirements.txt')
    print('directly into your Space\'s file browser on huggingface.co')

---
## Section 6 — 🟡 Go Further

**Challenge A — Add a RAG component:**  
Upload a text document → build a knowledge base → when an object is detected, retrieve related facts from the document and include them in the LLM response.

**Challenge B — Add streaming output:**  
```python
# Gradio supports streaming text output
def stream_response(image):
    for chunk in llm_pipe(prompt, stream=True):
        yield chunk

gr.Interface(fn=stream_response, outputs=gr.Textbox())
```

**Challenge C — Add a Chatbot component:**  
After detection, allow follow-up questions about the image using a chat history.

**Challenge D — Connect to an external API:**  
If a 'person' is detected → call a weather API and tell them what to wear.  
If 'car' detected → call a parking API. Any real-world data integration.

---
## Section 7 — Day 4 Recap

1. **What is Gradio?**  
   > A Python library that wraps any function into a web UI — no HTML/CSS/JS needed.

2. **What is a multi-model pipeline?**  
   > A system where the output of one model becomes the input of the next. Real AI products are almost always pipelines.

3. **What is HuggingFace Spaces?**  
   > A free hosting platform for AI demos. Upload your app.py and requirements.txt → get a permanent public URL.

4. **How did changing the LLM prompt change the app's behaviour?**  
   > Completely. Same models, same pipeline, different prompt = totally different product. Prompt engineering is product design.

5. **What are the failure modes of your pipeline?**  
   > YOLO misdetects → LLM responds to wrong labels. LLM hallucinates → wrong advice. No objects detected → empty response. Real systems need error handling at each step.

---

## 🚀 Tomorrow — Day 5: Capstone Project

You get 2 hours to build something of your own.  
**Requirements:** 2+ models · Gradio interface · Live demo (2 min)

> **Tonight:** Decide your project. Sketch the pipeline. Make sure your HF Space works.

**See you tomorrow — demo day! 🎯**